In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed_ames.csv')
print(df.shape)
df.head()

(2642, 227)


,MSSubClass,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_VWD,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,20,80,11622,4,4,3,5,6,1961,1961,...,0,0,0,0,1,0,0,0,1,0
1,20,81,14267,3,4,3,6,6,1958,1958,...,0,0,0,0,1,0,0,0,1,0
2,60,74,13830,3,4,3,5,5,1997,1998,...,0,0,0,0,1,0,0,0,1,0
3,60,78,9978,3,4,3,6,6,1998,1998,...,0,0,0,0,1,0,0,0,1,0
4,120,41,4920,4,4,3,8,5,2001,2001,...,0,0,0,0,1,0,0,0,1,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2

X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (2113, 226), Test: (529, 226)


In [3]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
df[X.columns].corrwith(np.log1p(df['SalePrice'])).sort_values(ascending=False).head(10)

TotalSF_Qual    0.873878
LivArea_Qual    0.853258
Bsmt_Qual       0.831103
OverallQual     0.820695
Bath_Qual       0.809243
Garage_Qual     0.799940
TotalSF         0.745551
GrLivArea       0.709648
GarageCars      0.684646
BsmtQual        0.681771
dtype: float64

In [6]:
from sklearn.linear_model import LinearRegression

# use best single feature from preprocessing correlation analysis
X_train_simple = X_train[['TotalSF_Qual']]
X_test_simple = X_test[['TotalSF_Qual']]

slr = LinearRegression()
slr.fit(X_train_simple, y_train_log)

y_pred_slr = slr.predict(X_test_simple)

rmse = np.sqrt(MSE(y_test_log, y_pred_slr))
mae = MAE(y_test_log, y_pred_slr)
r2 = R2(y_test_log, y_pred_slr)

print(f'Simple Linear Regression (TotalSF_Qual)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Simple Linear Regression (TotalSF_Qual)
RMSE : 0.1763
MAE  : 0.1340
R²   : 0.7525


### Model 1: Simple Linear Regression

A single-feature linear regression model was trained using `TotalSF_Qual` (total square
footage × overall quality) as the sole predictor, selected based on its highest
correlation (0.874) with the log-transformed target variable on the full dataset.

| Metric | Value |
|---|---|
| RMSE | 0.1763 |
| MAE  | 0.1340 |
| R²   | 0.7525 |

**Interpretation:**

- R² of 0.753 means a single feature explains ~75% of variance in sale price —
  a marginal improvement over the half-dataset baseline (0.717), explained by the
  larger and more representative sample stabilizing the regression line
- RMSE of 0.176 in log scale corresponds to roughly ±17% error in actual price terms
- `TotalSF_Qual` (total area × quality) overtook `LivArea_Qual` as the top feature
  on the full dataset, suggesting that total square footage captures more signal
  when the dataset includes a broader variety of property types

**Limitations:**

- A single feature cannot capture the full complexity of housing prices
- The model ignores 225 other features including neighborhood, age, condition,
  and structural characteristics
- Serves as the baseline — all subsequent models are evaluated against these metrics

In [7]:
mlr = LinearRegression()
mlr.fit(X_train, y_train_log)

y_pred_mlr = mlr.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_mlr))
mae = MAE(y_test_log, y_pred_mlr)
r2 = R2(y_test_log, y_pred_mlr)

print(f'Multiple Linear Regression')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Multiple Linear Regression
RMSE : 0.1064
MAE  : 0.0665
R²   : 0.9098


### Model 2: Multiple Linear Regression

Multiple Linear Regression was trained using all 226 features in the dataset,
extending the single-feature baseline to capture the full breadth of available
property information.

| Metric | SLR (Baseline) | MLR    | Change  |
|---|---|---|---|
| RMSE   | 0.1763         | 0.1064 | ↓ 40%   |
| MAE    | 0.1340         | 0.0665 | ↓ 50%   |
| R²     | 0.7525         | 0.9098 | ↑ 20.9% |

**Interpretation:**

- R² of 0.910 indicates the model explains ~91% of variance — a substantial jump
  over the half-dataset MLR (0.872), driven by the increased sample size (2113 vs 1046
  training samples) reducing the feature-to-sample ratio from ~1:5 to ~1:9
- RMSE of 0.106 in log scale corresponds to roughly ±11% error in actual price terms
- With more data, MLR fits the true signal more reliably and is less susceptible
  to being distorted by individual outlier observations

**Limitations:**

- At 226 features and 2113 training samples, overfitting risk has reduced but
  not disappeared — regularized models are still expected to outperform here
- Multicollinearity among engineered interaction features remains a concern,
  motivating Ridge and Lasso in the next steps

In [8]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

alphas = [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train, y_train_log, cv=5, scoring='r2')
    print(f'alpha={alpha:6} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

alpha=  0.01 | R²: 0.9060 ± 0.0176
alpha=   0.1 | R²: 0.9094 ± 0.0133
alpha=     1 | R²: 0.9137 ± 0.0126
alpha=    10 | R²: 0.9174 ± 0.0139
alpha=    50 | R²: 0.9142 ± 0.0146
alpha=   100 | R²: 0.9107 ± 0.0148
alpha=   200 | R²: 0.9064 ± 0.0150
alpha=   500 | R²: 0.8998 ± 0.0154
alpha=  1000 | R²: 0.8937 ± 0.0161


In [9]:
ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train_log)

y_pred_ridge = ridge.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_ridge))
mae = MAE(y_test_log, y_pred_ridge)
r2 = R2(y_test_log, y_pred_ridge)

print(f'Ridge Regression (alpha=10)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Ridge Regression (alpha=10)
RMSE : 0.0943
MAE  : 0.0654
R²   : 0.9292


### Model 3: Ridge Regression

Ridge Regression addresses overfitting and multicollinearity by adding an L2 penalty
term to the loss function, shrinking large coefficients toward zero without
eliminating features.

**Hyperparameter Tuning (Full Dataset):**

| Alpha  | CV R²  | Std Dev |
|---|---|---|
| 0.01   | 0.9060 | ±0.0176 |
| 0.1    | 0.9094 | ±0.0133 |
| 1      | 0.9137 | ±0.0126 |
| 10     | 0.9174 | ±0.0139 |
| 50     | 0.9142 | ±0.0146 |
| 100    | 0.9107 | ±0.0148 |
| 200    | 0.9064 | ±0.0150 |
| 500    | 0.8998 | ±0.0154 |
| 1000   | 0.8937 | ±0.0161 |

`alpha=10` remains optimal — same as the half-dataset, confirming the regularization
need is consistent. Notably, standard deviations are tighter across the board
(avg ±0.015 vs ±0.020 on half-dataset), reflecting more stable CV estimates on
larger data.

**Results:**

| Metric | SLR    | MLR    | Ridge  |
|---|---|---|---|
| RMSE   | 0.1763 | 0.1064 | 0.0943 |
| MAE    | 0.1340 | 0.0665 | 0.0654 |
| R²     | 0.7525 | 0.9098 | 0.9292 |

**Interpretation:**

- Ridge improves over MLR meaningfully here (R² 0.910 → 0.929), a larger margin
  than on the half-dataset (0.871 → 0.877) — more data amplifies the benefit
  of regularization by giving it more signal to work with
- CV std dev dropping to ±0.014 at alpha=10 confirms consistently stable
  generalization across folds
- Ridge is now a strong contender for best overall model in this pipeline

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
from sklearn.linear_model import Lasso

alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=20000)
    scores = cross_val_score(lasso, X_train_scaled, y_train_log, cv=5, scoring='r2')
    print(f'alpha={alpha:.4f} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

alpha=0.0001 | R²: 0.9118 ± 0.0127
alpha=0.0005 | R²: 0.9147 ± 0.0135
alpha=0.0010 | R²: 0.9166 ± 0.0140
alpha=0.0050 | R²: 0.9154 ± 0.0145
alpha=0.0100 | R²: 0.9065 ± 0.0142
alpha=0.0500 | R²: 0.8223 ± 0.0157
alpha=0.1000 | R²: 0.7255 ± 0.0110


In [12]:
lasso = Lasso(alpha=0.001, max_iter=10000)
lasso.fit(X_train_scaled, y_train_log)
y_pred_lasso = lasso.predict(X_test_scaled)

rmse = np.sqrt(MSE(y_test_log, y_pred_lasso))
mae = MAE(y_test_log, y_pred_lasso)
r2 = R2(y_test_log, y_pred_lasso)

zeroed = (lasso.coef_ == 0).sum()

print(f'Lasso Regression (alpha=0.005)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')
print(f'Features zeroed out: {zeroed} / {X_train.shape[1]}')

Lasso Regression (alpha=0.005)
RMSE : 0.0942
MAE  : 0.0637
R²   : 0.9293
Features zeroed out: 81 / 226


### Model 4: Lasso Regression

Lasso applies an L1 penalty which can drive coefficients to exactly zero,
performing automatic feature selection alongside regularization.

**Note on Feature Scaling:**

Lasso is trained on StandardScaler-transformed features. The coordinate descent
optimizer is sensitive to feature scale — scaling ensures the L1 penalty is applied
uniformly and prevents convergence issues.

**Hyperparameter Tuning (Full Dataset):**

| Alpha  | CV R²  | Std Dev |
|---|---|---|
| 0.0001 | 0.9118 | ±0.0127 |
| 0.0005 | 0.9147 | ±0.0135 |
| 0.0010 | 0.9166 | ±0.0140 |
| 0.0050 | 0.9154 | ±0.0145 |
| 0.0100 | 0.9065 | ±0.0142 |
| 0.0500 | 0.8223 | ±0.0157 |
| 0.1000 | 0.7255 | ±0.0110 |

`alpha=0.001` was selected — highest CV R² (0.9166) with stable std dev.
This is a shift from `alpha=0.005` on the half-dataset, reflecting that with
more data, less regularization is needed — the model can rely on actual signal
rather than requiring heavy penalty to avoid overfitting noise.

**Results:**

| Metric | SLR    | MLR    | Ridge  | Lasso  |
|---|---|---|---|---|
| RMSE   | 0.1763 | 0.1064 | 0.0943 | 0.0942 |
| MAE    | 0.1340 | 0.0665 | 0.0654 | 0.0637 |
| R²     | 0.7525 | 0.9098 | 0.9292 | 0.9293 |
| Features zeroed | — | — | — | 81 / 226 |

**Interpretation:**

- Lasso matches Ridge almost exactly on test metrics (R² 0.9293 vs 0.9292)
- Critically, Lasso only zeroed out 81/226 features (36%) vs 158/219 (72%) on the
  half-dataset — with more data, more features carry reliable signal and Lasso
  retains them instead of eliminating them
- The 145 retained features represent the model's learned sparse view of what
  actually drives house prices in Ames

In [13]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train[['TotalSF_Qual']])
X_test_poly = poly.transform(X_test[['TotalSF_Qual']])

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train_log)
y_pred_poly = poly_model.predict(X_test_poly)

rmse = np.sqrt(MSE(y_test_log, y_pred_poly))
mae = MAE(y_test_log, y_pred_poly)
r2 = R2(y_test_log, y_pred_poly)

print(f'Polynomial Regression (degree=2, TotalSF_Qual)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Polynomial Regression (degree=2, TotalSF_Qual)
RMSE : 0.1745
MAE  : 0.1326
R²   : 0.7575


In [14]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

max_depths = [3, 5, 7, 10, 15, 20, None]

for depth in max_depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(dt, X_train, y_train_log, cv=5, scoring='r2')
    print(f'max_depth={str(depth):5} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

max_depth=3     | R²: 0.7521 ± 0.0185
max_depth=5     | R²: 0.7781 ± 0.0333
max_depth=7     | R²: 0.7879 ± 0.0194
max_depth=10    | R²: 0.7813 ± 0.0126
max_depth=15    | R²: 0.7620 ± 0.0175
max_depth=20    | R²: 0.7669 ± 0.0234
max_depth=None  | R²: 0.7611 ± 0.0261


In [15]:
dt = DecisionTreeRegressor(max_depth=7, random_state=42)
dt.fit(X_train, y_train_log)
y_pred_dt = dt.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_dt))
mae = MAE(y_test_log, y_pred_dt)
r2 = R2(y_test_log, y_pred_dt)

print(f'Decision Tree (max_depth=5)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Decision Tree (max_depth=5)
RMSE : 0.1667
MAE  : 0.1193
R²   : 0.7787


### Model 5: Decision Tree Regressor

Decision trees recursively split data on feature thresholds, learning a set of
if-else rules mapping features to predicted outcomes.

**Hyperparameter Tuning (Full Dataset):**

| max_depth | CV R²  | Std Dev |
|---|---|---|
| 3         | 0.7521 | ±0.0185 |
| 5         | 0.7781 | ±0.0333 |
| 7         | 0.7879 | ±0.0194 |
| 10        | 0.7813 | ±0.0126 |
| 15        | 0.7620 | ±0.0175 |
| 20        | 0.7669 | ±0.0234 |
| None      | 0.7611 | ±0.0261 |

`max_depth=7` selected — best CV R² (0.788) with low std dev. This is deeper
than the half-dataset optimum of 5, as the larger dataset supports more granular
splits without overfitting to noise.

**Results:**

| Metric | SLR    | MLR    | Ridge  | Lasso  | Decision Tree |
|---|---|---|---|---|---|
| RMSE   | 0.1763 | 0.1064 | 0.0943 | 0.0942 | 0.1667        |
| MAE    | 0.1340 | 0.0665 | 0.0654 | 0.0637 | 0.1193        |
| R²     | 0.7525 | 0.9098 | 0.9292 | 0.9293 | 0.7787        |

**Interpretation:**

- Decision Tree improved (R² 0.760 → 0.779) and CV variance dropped sharply
  (±0.07 → ±0.019) — the key gain from more data is stability, not accuracy
- Still significantly behind all linear models, confirming a single tree cannot
  match the generalization of regularized regression on this dataset
- The improvement with deeper splits (depth 7 vs 5) shows the tree is now learning
  more meaningful patterns rather than hitting its complexity ceiling early
- This motivates ensemble methods: Random Forest and XGBoost aggregate many trees
  to overcome the single-tree limitation

In [16]:
from sklearn.ensemble import RandomForestRegressor

n_estimators = [50, 100, 200]
max_depths = [10, 20, None]

for n in n_estimators:
    for depth in max_depths:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=42, n_jobs=-1)
        scores = cross_val_score(rf, X_train, y_train_log, cv=5, scoring='r2')
        print(f'n={n:3}, depth={str(depth):5} | R²: {scores.mean():.4f} ± {scores.std():.4f}')

n= 50, depth=10    | R²: 0.8819 ± 0.0085
n= 50, depth=20    | R²: 0.8844 ± 0.0090
n= 50, depth=None  | R²: 0.8845 ± 0.0086
n=100, depth=10    | R²: 0.8844 ± 0.0083
n=100, depth=20    | R²: 0.8871 ± 0.0088
n=100, depth=None  | R²: 0.8872 ± 0.0084
n=200, depth=10    | R²: 0.8859 ± 0.0079
n=200, depth=20    | R²: 0.8886 ± 0.0085
n=200, depth=None  | R²: 0.8887 ± 0.0083


In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train_log)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV R²: {grid_search.best_score_:.4f}')

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV R²: 0.8907


In [18]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train_log)
y_pred_rf = rf.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_rf))
mae = MAE(y_test_log, y_pred_rf)
r2 = R2(y_test_log, y_pred_rf)

print(f'Random Forest (tuned)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

Random Forest (tuned)
RMSE : 0.1113
MAE  : 0.0763
R²   : 0.9013


### Model 6: Random Forest Regressor

Random Forest builds an ensemble of decorrelated decision trees using bagging
(random row sampling) and feature subsampling at each split, averaging predictions
to reduce variance.

**Grid Search Best Params:** `n_estimators=200, max_depth=15, max_features='sqrt',
min_samples_leaf=1, min_samples_split=2` | Best CV R²: **0.8907**

**Results:**

| Metric | SLR    | MLR    | Ridge  | Lasso  | DT     | RF     |
|---|---|---|---|---|---|---|
| RMSE   | 0.1763 | 0.1064 | 0.0943 | 0.0942 | 0.1667 | 0.1113 |
| MAE    | 0.1340 | 0.0665 | 0.0654 | 0.0637 | 0.1193 | 0.0763 |
| R²     | 0.7525 | 0.9098 | 0.9292 | 0.9293 | 0.7787 | 0.9013 |

**Interpretation:**

- Random Forest improves substantially over the half-dataset (R² 0.840 → 0.901)
  — the largest relative gain among all models, confirming that tree ensembles
  benefit most from additional data
- Optimal depth shifted from 20 to 15: with more data per leaf, shallower trees
  generalize better and avoid over-partitioning
- Despite the improvement, RF (0.901) still trails Ridge/Lasso (0.929), suggesting
  that regularized linear models remain more efficient on this structured tabular
  dataset at this scale

In [19]:
import xgboost as xgb

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
}

xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
grid_search_xgb = GridSearchCV(xgb_model, params, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search_xgb.fit(X_train, y_train_log)

print(f'Best params: {grid_search_xgb.best_params_}')
print(f'Best CV R²: {grid_search_xgb.best_score_:.4f}')

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best params: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.8}
Best CV R²: 0.9119


In [20]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train_log)
y_pred_xgb = xgb_model.predict(X_test)

rmse = np.sqrt(MSE(y_test_log, y_pred_xgb))
mae = MAE(y_test_log, y_pred_xgb)
r2 = R2(y_test_log, y_pred_xgb)

print(f'XGBoost (tuned)')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

XGBoost (tuned)
RMSE : 0.0996
MAE  : 0.0680
R²   : 0.9210


### Model 7: XGBoost

XGBoost builds trees sequentially, with each tree correcting the residual errors
of the previous one (boosting). Combined with regularization and subsampling,
it is typically the strongest single model on structured tabular data.

**Grid Search Best Params:** `n_estimators=300, max_depth=4, learning_rate=0.05,
subsample=0.8` | Best CV R²: **0.9119**

**Results:**

| Metric | SLR    | MLR    | Ridge  | Lasso  | DT     | RF     | XGBoost |
|---|---|---|---|---|---|---|---|
| RMSE   | 0.1763 | 0.1064 | 0.0943 | 0.0942 | 0.1667 | 0.1113 | 0.0996  |
| MAE    | 0.1340 | 0.0665 | 0.0654 | 0.0637 | 0.1193 | 0.0763 | 0.0680  |
| R²     | 0.7525 | 0.9098 | 0.9292 | 0.9293 | 0.7787 | 0.9013 | 0.9210  |

**Interpretation:**

- XGBoost improved over half-dataset (R² 0.862 → 0.921) and overtook RF clearly
- Optimal `max_depth` shifted from 3 to 4: more data allows slightly deeper trees
  before overfitting kicks in
- XGBoost (0.921) still falls short of Ridge/Lasso (0.929) on this dataset —
  an important finding: on clean, well-engineered tabular data with strong linear
  relationships, regularized linear models can match or beat gradient boosting

---

## Final Model Comparison Summary

| Model         | RMSE   | MAE    | R²     |
|---|---|---|---|
| SLR           | 0.1763 | 0.1340 | 0.7525 |
| Poly Reg      | 0.1745 | 0.1326 | 0.7575 |
| MLR           | 0.1064 | 0.0665 | 0.9098 |
| Decision Tree | 0.1667 | 0.1193 | 0.7787 |
| Random Forest | 0.1113 | 0.0763 | 0.9013 |
| XGBoost       | 0.0996 | 0.0680 | 0.9210 |
| Ridge         | 0.0943 | 0.0654 | 0.9292 |
| **Lasso**     | **0.0942** | **0.0637** | **0.9293** |

**Winner: Lasso Regression (alpha=0.001)**

Lasso edges out Ridge marginally and outperforms XGBoost — a result that reflects
the quality of feature engineering done in preprocessing. A well-engineered feature
set reduces the advantage that boosting models typically hold on raw data.